In [1]:
import pandas as pd
import numpy as np

LOAD DATA

In [2]:
df=pd.read_csv("/content/Logistics_Delivery.csv")
print("Original Dataset")
print(df.head())
print("\nShape:",df.shape)

Original Dataset
  Shipment_ID           Order_Date            Ship_Date        Promised_Date  \
0   PKG-50637  2024-03-07 00:00:00  2024-03-07 00:00:00  2024-03-10 00:00:00   
1   PKG-50895  2024-03-05 00:00:00  2024-03-05 00:00:00  2024-03-06 00:00:00   
2   PKG-50300           29/12/2025           30/12/2025  2026-01-02 00:00:00   
3   PKG-50107  2024-09-16 00:00:00  2024-09-16 00:00:00  2024-09-19 00:00:00   
4   PKG-51087  2024-09-01 00:00:00  2024-09-04 00:00:00  2024-09-05 00:00:00   

         Delivery_Date Origin_City Destination_City Destination_Zone  \
0  2024-03-09 00:00:00     Chennai             Pune             West   
1  2024-03-07 00:00:00     Chennai          Chennai            South   
2  2023-12-15 00:00:00        Pune       Trivandrum            South   
3  2024-09-19 00:00:00       Delhi          Chennai            South   
4  2024-09-06 00:00:00         NaN           Mumbai             West   

  Warehouse    Carrier Delivery_Mode Customer_Name Customer_Type  \
0

BEFORE CLEANING ANALYSIS

In [5]:
before_rows=len(df)
before_columns=len(df.columns)
print("\n---BEFORE CLEANING---")
print("Rows:",before_rows)
print("Columns:",before_columns)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:",df.duplicated().sum())
print("\nDuplicate Shipment IDs:",
      df["Shipment_ID"].duplicated().sum())


---BEFORE CLEANING---
Rows: 1630
Columns: 20

Missing Values:
Shipment_ID           17
Order_Date             0
Ship_Date             18
Promised_Date         20
Delivery_Date        236
Origin_City           17
Destination_City      17
Destination_Zone       0
Warehouse             17
Carrier               17
Delivery_Mode          0
Customer_Name          0
Customer_Type         13
Package_Weight_Kg     34
Distance_Km           27
Shipping_Cost         21
Order_Value           22
Payment_Method         0
Delivery_Status        0
Vehicle_Type          13
dtype: int64

Duplicate Rows: 30

Duplicate Shipment IDs: 68


REMOVE EXTRA SPACES FROM TEXT COLUMNS

In [6]:
text_columns=df.select_dtypes(include="object").columns
for col in text_columns:
  df[col]=df[col].astype("string").str.strip()

STANDARDIZE TEXT VALUES

In [7]:
df["Delivery_Mode"]=(
    df["Delivery_Mode"]
    .str.lower()
    .replace({
        "same-day": "Same Day",
        "same day": "Same Day",
        "express delivery": "Express",
        "express": "Express",
        "standard": "Standard",
        "economical": "Economy",
        "economy": "Economy",

    })
)

df["Payment_Method"]=(
    df["Payment_Method"]
    .str.lower()
    .str.strip()
    .replace({
        "cod":" COD",
        "creditcard": "Credit Card",
        "credit card": "Credit Card",
        "pre-paid": "prepaid",
        "upi":"UPI"


    })
)

df["Delivery_Status"]=(
    df["Delivery_Status"]
    .str.lower()
    .str.strip()
    .replace({
        "delivered": "Delivered",
        "delayed": "Delayed",
        "cancelled": "Cancelled",
        "returned": "In Transit",
        "in-transit": "In Transit"
    })
)

STANDARDIZE CITY NAMES

In [9]:
city_mapping={
    "bangalore": "Bangaluru",
    "bengaluru": "Bengaluru",
    "cochin": "Kochi",
    "thiruvanathapuram": "Trivandrum",
    "trivandrum": "Trivandrum",
    "bombay": "Mumbai",
    "mumbali": "Mumbai"

}
for col in ["Origin_City","Destination_City"]:
  df[col]=(
      df[col]
      .str.lower()
      .replace(city_mapping)
  )

CONVERT NUMERIC COLUMNS

In [12]:
numeric_columns=[
    "Package_Weight_Kg",
    "Distance_Km",
    "Shipping_Cost",
    "Order_Value"
]

for col in numeric_columns:
  df[col]=(
      df[col]
      .astype(str)
      .str.replace(",", "", regex=False)
      .str.extract(r"(-?\d+\.?\d*)")[0]
      .astype(float)
  )

CONVERT DATE COLUMNS

In [16]:
date_columns=[
    "Order_Date",
    "Ship_Date",
    "Promised_Date",
    "Delivery_Date"
]

for col in date_columns:
  df[col]=pd.to_datetime(
      df[col],
      errors="coerce",
      dayfirst=True
  )

CHECK OUTLIERS USING IQR

In [17]:
def find_outlier(date,column):
  Q1=data[column].quantile(0.25)
  Q3=data[column].quantile(0.75)

  IQR=Q3-Q1

  lower_limit=Q3 - 1.5 * IQR
  upper_limit= Q3 + 1.5 * IQR

  outliers=data[
      (data[column]< lower_limit) |
      (data[column] > upper_limit)
  ]
  return outliers, lower_limit, upper_limit

  print("\n---OUTLIER ANALYSIS ---")
  for col in numeric_columns:

    outlier,lower,upper =find_outliers(df,col)

    print (f"\nColumn: {col}")
    print("Lower Limit:", upper)
    print("Number of Outliers:" , len(outliers))

HANDLE EXTREME OUTLIERS

In [18]:
for col in numeric_columns:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)

  IQR=Q3-Q1
  lower_limit = Q1-3 * IQR
  upper_limit=Q3 + 3* IQR
  df[col]=df[col].clip(
      lower=lower_limit,
      upper=upper_limit
  )

REMOVE DUPLICATE ROWS

In [20]:
duplicate_rows=df.duplicated().sum()
df=df.drop_duplicates()
print("\nDuplicate rows removed:",duplicate_rows)


Duplicate rows removed: 30


REMOVE DUPLICATE SHIPMENT IDs

In [21]:
df=(
    df.sort_values("Shipment_ID")
    .drop_duplicates(
        subset="Shipment_ID",
        keep="first"
    )
)

HANDLE MISSING VALUES

In [25]:
for col in numeric_columns:
  df[col]=df[col].fillna(
      df[col].median()
  )

  categorical_columns=[
      "Origin_City",
      "Destination_City",
      "Destination_Zone",
      "Warehouse",
      "Carrier",
      "Delivery_Mode",
      "Customer_Type",
      "Payment_Method",
      "Delivery_Status",
      "Vehicle_Type"
  ]

  for col in categorical_columns:
    if df[col].isnull().sum() > 0:
      mode_value=df[col].mode()[0]
      df[col]=df[col].fillna(
          mode_value
      )

      df["Customer_Name"]=df["Customer_Name"].fillna(
          "Unknown Customer"
      )

FIX DATE LOGIC

In [26]:
invalid_ship=(
    df["Ship_Date"].notna() &
    df["Order_Date"].notna() &
    (df["Ship_Date"] <df["Order_Date"])

)
df.loc[invalid_ship, "Ship_Date"]=pd.NaT

invalid_promised=(
    df["Promised_Date"].notna() &
    df["Order_Date"].notna() &
    (df["Promised_Date"] < df["Order_Date"])
)
df.loc[invalid_promised, "Promised_Date"]=pd.NaT

invalid_delivery=(
    df["Delivery_Date"].notna() &
    df["Ship_Date"].notna() &
    (df["Delivery_Date"] < df["Ship_Date"])
)
df.loc[invalid_delivery, "Delivery_Date"]=pd.NaT


CREATE NEW ANALYTICAL COLUMNS

In [29]:
df["Transit_Days"]=(
    df["Delivery_Date"] -
    df["Ship_Date"]

).dt.days

df["Delivery_Delay_Days"]=(
    df["Delivery_Date"] -
    df["Promised_Date"]
).dt.days

FINAL DATA QUALITY CHECK

In [32]:
after_rows=len(df)
after_columns=len(df.columns)

print("\n===================================")
print("FINAL DATA QUALITY REPORT")
print("==================================")
print("\nRows Before:" , before_rows)
print("Rows After:", after_rows)
print("Columns After:", after_columns)
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())
print("\nDuplicates Shipment IDs After:",
      df["Shipment_ID"].duplicated().sum())


FINAL DATA QUALITY REPORT

Rows Before: 1630
Rows After: 1562
Columns After: 22

Missing Values After Cleaning:
Shipment_ID               1
Order_Date             1011
Ship_Date              1000
Promised_Date          1012
Delivery_Date          1085
Origin_City               0
Destination_City          0
Destination_Zone          0
Warehouse                 0
Carrier                   0
Delivery_Mode             0
Customer_Name             0
Customer_Type             0
Package_Weight_Kg         0
Distance_Km               0
Shipping_Cost             0
Order_Value               0
Payment_Method            0
Delivery_Status           0
Vehicle_Type              0
Transit_Days           1269
Delivery_Delay_Days    1195
dtype: int64

Duplicates Shipment IDs After: 0


BEFORE VS AFTER COMPARISON

In [35]:
comaprison=pd.DataFrame({
    "Metric":[
    "Rows",
        "Columns",
        "Duplicate Rows",
        "Duplicate Shipment IDs",
        "Missing Values"
    ],
    "Before":[
        before_rows,
        before_columns,
        duplicate_rows,
        df["Shipment_ID"].duplicated().sum(),
        pd.read_csv("/content/Logistics_Delivery.csv").isnull().sum().sum()
    ],
    "After":[
        after_rows,
        after_columns,
        df.duplicated().sum(),
        df["Shipment_ID"].duplicated().sum(),
        df.isnull().sum().sum()
    ]
})
print("\n--- BEFORE VS AFTER ---")
print(comaprison)


--- BEFORE VS AFTER ---
                   Metric  Before  After
0                    Rows    1630   1562
1                 Columns      20     22
2          Duplicate Rows      30      0
3  Duplicate Shipment IDs       0      0
4          Missing Values     489   6573


SAVE CLEANED DATA

In [38]:
df.to_csv(
    "Logistics_Delivery_Cleaned.csv",
    index=False
)
comaprison.to_csv(
    "Before_After_Comparison.csv",
    index=False
)
print("\nCleaned dataset saved successfully")
print("File: Logistics_Delivery_Cleaned.csv")
print("Comparison saved as: Before_After_Comparison.csv")


Cleaned dataset saved successfully
File: Logistics_Delivery_Cleaned.csv
Comparison saved as: Before_After_Comparison.csv
